# Kaggle Ollama Server

Run this notebook on Kaggle (GPU T4 x2 or P100 — both work).

It will:
1. Install Ollama
2. Pull `qwen2.5:3b`
3. Expose the API publicly via ngrok
4. Print the URL you paste into Molab

**Keep this notebook running the whole time your Molab pipeline runs.**

### Setup (one-time)
1. Go to https://dashboard.ngrok.com → sign up free → copy your **authtoken**
2. In Kaggle → Add-ons → Secrets → add secret named `NGROK_TOKEN` with your token
3. Run all cells

In [ ]:
# Cell 1 — Install dependencies
import os, subprocess, time, threading

print('Installing zstd and ngrok...')
subprocess.run('apt-get install -y -qq zstd > /dev/null 2>&1', shell=True)
subprocess.run('pip install -q pyngrok', shell=True)
print('Done.')

In [ ]:
# Cell 2 — Install Ollama
os.environ['PATH'] = '/usr/local/bin:/usr/bin:/bin:' + os.environ.get('PATH', '')

if subprocess.run('which ollama', shell=True, capture_output=True).returncode != 0:
    print('Installing Ollama...')
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True)
else:
    print('Ollama already installed.')

ollama_bin = '/usr/local/bin/ollama'
if not os.path.isfile(ollama_bin):
    r = subprocess.run('which ollama', shell=True, capture_output=True, text=True)
    ollama_bin = r.stdout.strip() if r.returncode == 0 else 'ollama'
print(f'Ollama binary: {ollama_bin}')

In [ ]:
# Cell 3 — Start Ollama server
subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
time.sleep(2)

os.makedirs('/tmp/ollama_logs', exist_ok=True)
subprocess.Popen(
    f'{ollama_bin} serve >> /tmp/ollama_logs/serve.log 2>&1',
    shell=True
)

print('Waiting for Ollama API...')
for i in range(20):
    time.sleep(2)
    r = subprocess.run('curl -sf http://localhost:11434/api/tags', shell=True, capture_output=True)
    if r.returncode == 0:
        print(f'Ollama running (took {(i+1)*2}s)')
        break
else:
    print('WARNING: Ollama slow — check /tmp/ollama_logs/serve.log')

In [ ]:
# Cell 4 — Pull model
MODEL = 'qwen2.5:3b'
print(f'Pulling {MODEL} ...')
r = subprocess.run(f'{ollama_bin} pull {MODEL}', shell=True)
if r.returncode == 0:
    print(f'Model {MODEL} ready')
else:
    print('Pull failed — trying llama3.2:3b as fallback')
    subprocess.run(f'{ollama_bin} pull llama3.2:3b', shell=True)
    MODEL = 'llama3.2:3b'

print(f'Active model: {MODEL}')

In [ ]:
# Cell 5 — Expose via ngrok
from pyngrok import ngrok, conf

# Read ngrok token from Kaggle secret
try:
    from kaggle_secrets import UserSecretsClient
    NGROK_TOKEN = UserSecretsClient().get_secret('NGROK_TOKEN')
except Exception:
    # Fallback: paste your token directly here if not using Kaggle secrets
    NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'

if 'PASTE_YOUR' in NGROK_TOKEN:
    raise ValueError(
        'Set your ngrok token!\n'
        'Option A: Kaggle Secrets → add NGROK_TOKEN\n'
        'Option B: Paste token directly into NGROK_TOKEN variable above'
    )

conf.get_default().auth_token = NGROK_TOKEN
# Kill any existing tunnels
ngrok.kill()
time.sleep(1)

# Open HTTP tunnel to Ollama port
tunnel = ngrok.connect(11434, 'http')
public_url = tunnel.public_url

print('=' * 60)
print('OLLAMA SERVER IS LIVE')
print('=' * 60)
print(f'Public URL: {public_url}')
print()
print('Paste this into Molab molab_run.py:')
print(f'    OLLAMA_BASE_URL = "{public_url}"')
print()
print('Keep this notebook running while Molab pipeline is running.')
print('=' * 60)

In [ ]:
# Cell 6 — Keep-alive (MUST keep running)
# This cell keeps the notebook alive and watchdogs Ollama.
# The Kaggle session times out after ~9 hours (GPU) or 12 hours (CPU).
import requests

print('Keep-alive started. Heartbeat every 5 min.')
print(f'Ollama URL: {public_url}')
print()

counter = 0
while True:
    time.sleep(300)
    counter += 1
    h = counter * 5 / 60

    # Check Ollama health
    try:
        resp = requests.get('http://localhost:11434/api/tags', timeout=5)
        ollama_ok = resp.status_code == 200
    except Exception:
        ollama_ok = False

    if not ollama_ok:
        print(f'[{h:.1f}h] Ollama died — restarting...')
        subprocess.Popen(
            f'{ollama_bin} serve >> /tmp/ollama_logs/serve.log 2>&1',
            shell=True
        )
        time.sleep(10)
    else:
        # Check tunnel still live
        try:
            tunnels = ngrok.get_tunnels()
            tunnel_ok = len(tunnels) > 0
        except Exception:
            tunnel_ok = False

        if not tunnel_ok:
            print(f'[{h:.1f}h] Tunnel died — restarting...')
            tunnel = ngrok.connect(11434, 'http')
            public_url = tunnel.public_url
            print(f'[{h:.1f}h] New URL: {public_url}')
        else:
            print(f'[{h:.1f}h] OK — Ollama running | Tunnel: {public_url}')